In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_silver = spark.sql('''
                      SELECT * FROM PARQUET.`abfss://silver@storageforcarsproject.dfs.core.windows.net/OBT`''')
display(df_silver)

In [0]:
df_model = spark.sql("SELECT * FROM cars_cata.gold.dim_model")
df_branch = spark.sql("SELECT * FROM cars_cata.gold.dim_branch")
df_dealer = spark.sql("SELECT * FROM cars_cata.gold.dim_dealer")
df_date = spark.sql("SELECT * FROM cars_cata.gold.dim_date")



In [0]:
df_fact = df_silver.join(df_branch,df_silver["Branch_ID"]==df_branch["Branch_ID"],"LEFT")\
        .join(df_dealer,df_silver["Dealer_ID"]==df_dealer["Dealer_ID"],"LEFT")\
        .join(df_model,df_silver["Model_ID"]==df_model["Model_ID"],"LEFT")\
        .join(df_date,df_silver["Date_ID"]==df_date["Date_ID"],"LEFT")\
        .select(df_silver["Revenue"],df_silver["Units_Sold"],df_silver["cost_per_unit"],df_model["dim_model_key"],df_branch["dim_branch_key"],df_dealer["dim_dealer_key"],df_date["dim_date_key"])
        

In [0]:
display(df_fact)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_cata.gold.factsales'): 
    deltatable = DeltaTable.forName(spark, 'cars_cata.gold.factsales')

    deltatable.alias('trg').merge(df_fact.alias('src'), 'trg.dim_branch_key = src.dim_branch_key and trg.dim_dealer_key = src.dim_dealer_key and trg.dim_model_key = src.dim_model_key and trg.dim_date_key = src.dim_date_key')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else: 
    df_fact.write.format('delta')\
            .mode('overwrite')\
            .option("path", "abfss://gold@storageforcarsproject.dfs.core.windows.net/factsales")\
            .saveAsTable('cars_cata.gold.factsales')

In [0]:
%sql
select * from cars_cata.gold.factsales

In [0]:
%sql
drop table cars_cata.gold.factsales